# LaLonde Dataset: Propensity Score Matching with CausalML

This notebook demonstrates propensity score matching on the LaLonde dataset to estimate the average treatment effect on the treated (ATT) of job training on earnings.

We use **observational data** (experimental treated units + non-experimental PSID control group) which has selection bias, and compare our estimate against the true experimental ATT from the randomized trial.

In [4]:
# Add project root to path so we can import the datasets module
import sys
from pathlib import Path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [5]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from causalml.match import NearestNeighborMatch
from datasets import LalondeDataset

In [53]:
# Load LaLonde dataset with observational data
# This combines experimental treated (185) with non-experimental PSID controls (2,490)
ds = LalondeDataset()
data = ds.observational()

print(f"Dataset shape: {data.shape}")
print(f"\nColumns: {list(data.columns)}")
print(f"\nTreated: {data.treat.sum()}")

print(f"\nFirst few rows:")
data.head()

Dataset shape: (2675, 12)

Columns: ['treat', 'age', 'educ', 'black', 'hisp', 'married', 'nodegr', 're74', 're75', 're78', 'u74', 'u75']

Treated: 185

First few rows:


,treat,age,educ,black,hisp,married,nodegr,re74,re75,re78,u74,u75
0,True,37.0,11.0,1.0,0.0,1.0,1.0,0.0,0.0,9930.0460,1.0,1.0
1,True,22.0,9.0,0.0,1.0,0.0,1.0,0.0,0.0,3595.8940,1.0,1.0
2,True,30.0,12.0,1.0,0.0,0.0,0.0,0.0,0.0,24909.4500,1.0,1.0
3,True,27.0,11.0,1.0,0.0,0.0,1.0,0.0,0.0,7506.1460,1.0,1.0
4,True,33.0,8.0,1.0,0.0,0.0,1.0,0.0,0.0,289.7899,1.0,1.0


In [57]:
# Define treatment, outcome, and covariates
treatment_col = "treat"
outcome_col = "re78"
covariate_cols = ["age", "educ", "black", "hisp", "married", "nodegr", "re74", "re75"]

# Fit propensity scores using logistic regression
X = data[covariate_cols]
treatment = data[treatment_col]

psmodel = LogisticRegression(max_iter=3000, random_state=42, fit_intercept=True)
psmodel.fit(X, treatment)
data["propensity_score"] = psmodel.predict_proba(X)[:, 1]

# Perform nearest neighbor matching on propensity scores
matcher = NearestNeighborMatch(caliper=0.05, random_state=42)
matched_data = matcher.match(data=data, treatment_col=treatment_col, score_cols=["propensity_score"])

print(f"Original data: {len(data)} rows")
print(f"Matched data: {len(matched_data)} rows")

Original data: 2675 rows
Matched data: 160 rows


In [58]:
# Compute Average Treatment Effect on the Treated (ATT)
treated_outcome = matched_data.loc[matched_data[treatment_col] == 1, outcome_col].mean()
control_outcome = matched_data.loc[matched_data[treatment_col] == 0, outcome_col].mean()
att = treated_outcome - control_outcome

print(f"\nAverage outcome for treated: ${treated_outcome:,.2f}")
print(f"Average outcome for control: ${control_outcome:,.2f}")
print(f"\nEstimated ATT: ${att:,.2f}")


Average outcome for treated: $6,839.69
Average outcome for control: $8,187.68

Estimated ATT: $-1,347.99


In [59]:
# Compare with true experimental effect
# The true ATT comes from the randomized experiment (NSW demonstration)
print(f"\n{'='*60}")
print(f"Comparison with Experimental Benchmark")
print(f"{'='*60}")
print(f"True experimental ATT: ${ds.true_att:,.2f}")
print(f"PSM estimated ATT:     ${att:,.2f}")
print(f"Difference:            ${att - ds.true_att:,.2f} ({(att - ds.true_att) / ds.true_att * 100:.1f}%)")


Comparison with Experimental Benchmark
True experimental ATT: $1,794.34
PSM estimated ATT:     $-1,347.99
Difference:            $-3,142.33 (-175.1%)
